# 03 | 指標の整合性確認・レポート下書き・提出物チェック
01と02の出力CSV/JSONを `outputs` ディレクトリにアップロードして実行。
**2D計画評価と3D物理走行は異なる実験なので統合して1つの物理性能指標と呼ばない**。

In [3]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
out = Path('outputs')
out.mkdir(exist_ok = True)
print([str(p) for p in out.iterdir()])

['outputs/genesis_physical_metrics.json']


In [4]:
plan_path = out/'planner_only_metrics.csv'
physics_path = out/'genesis_physical_metrics.json'
assert physics_path.exists(),'First complete 02 and transfer outputs/genesis_physical_metrics.json'
phys = json.loads(physics_path.read_text(encoding = 'utf8'))
print('Genesis physical run:')
display(pd.DataFrame([phys]).T)
if plan_path.exists():
    df = pd.read_csv(plan_path)
    print('Separate planner-only comparison:')
    display(df.groupby('policy')[['recall','weighted_detection_delay_s','distance_m','coverage']].mean())
else:
    print('01 planner-only evaluation not supplied; do not report unmeasured improvement.')

Genesis physical run:


,0
policy,adaptive
seed,42
simulated_3d_seconds,90.0
physical_distance_m,7.403478
inspected_stations,0
replans,1
geofence_proximity_steps_proxy,0
below_floor_steps,0
video,outputs/genesis_adaptive.mp4
method,Genesis wheel-joint velocity control and base ...


01 planner-only evaluation not supplied; do not report unmeasured improvement.


In [5]:
report=f'''# AI異常検知とリスク適応巡回を組み合わせた3D点検ロボット

## 1. 目的と概要
設備の合成テレメトリから異常度を推定し、点検優先度を動的に変える巡回方式を検討した。

## 2. システム構成
Genesis World 1.4.4の3D剛体シミュレータ、自作差動二輪URDF、車輪関節の速度制御、ロボット位置の閉ループ観測、Isolation Forest、A*経路探索。

## 3. AIの役割と工夫
故障時刻・正解IDを与えずに合成観測からスコアを算出。固定順でなく異常度と距離を使って点検対象を変更。正常学習・比較条件の固定。

## 4. 実行結果（3Dの実測）
- シミュレーション長: {phys['simulated_3d_seconds']} s
- 3D物理位置から求めた移動距離: {phys['physical_distance_m']:.2f} m
- 実地（シミュレーション上）点検回数: {phys['inspected_stations']}
- 再計画回数: {phys['replans']}
- ジオフェンス近接検出ステップ: {phys['geofence_proximity_steps_proxy']}（**衝突件数ではない**）

## 5. 比較実験（2D運動学の別実験）
別Notebookの同一シード対比較で評価。ここへ**実測値**を追記する。2DとGenesis 3Dの運動性能は直接比較しない。

## 6. 限界・展望
観測値は合成センサーで、実設備・実センサーは用いていない。ロボットの力学パラメータは簡易URDFモデル。SIM-to-Real妥当性は未検証。EGL/Colab GPUリソースの差異による再現性課題あり。

## 7. 再現性
実行Notebook、生成URDF、ランタイム依存関係、乱数シード、ログを公開。個人情報・APIキーなし。
'''
(out/'report_draft_genesis.md').write_text(report, encoding = 'utf8')
print('Generated', out/'report_draft_genesis.md')

Generated outputs/report_draft_genesis.md


In [6]:
import subprocess
movie = out/'genesis_adaptive.mp4'
if movie.exists():
    cmd = ['ffprobe','-v','error','-select_streams','v:0','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1', str(movie)]
    p = subprocess.run(cmd, capture_output = True, text = True)
    if p.returncode == 0:
        secs = float(p.stdout.strip())
        print('Movie duration:', secs)
        assert 60 <= secs <= 180
    else:
        print('ffprobe failed:', p.stderr)
else:
    print('MP4 not transferred; submit only after verified.')

MP4 not transferred; submit only after verified.


## 提出先チェック
- [ ] PDF（個人情報なし、OmniCampus指定名）
- [ ] GitHub public（Notebook+src+README。非公開情報、APIキーなし）
- [ ] 1〜3分MP4 をYouTubeにアップ（最低限限定公開、子ども向け・年齢制限は「いいえ」）
- [ ] URLをOmniCampus提出。**2026/11/02 10:00 JST締切**。
- [ ] 実際の3D走行、AI推論、比較実験の方法・限界が動画とPDFで区別されている